# 07 Sentinel-1 SLC Data Acquisition from CDSE

**Author:** Florian Klaver

In this Notebook a different SAR product is acquired, namely the raw SLC from CDSE.


### Why SLC? Why coherence?

Notebook 04 computes **backscatter** features from GRD products. The GRD-only model achieved F1 ≈ 0.5 (barely above random), showing that GRD backscatter alone is a weak and noisy mowing signal.


**InSAR coherence** is fundamentally different: it measures the *phase correlation* between two SAR acquisitions from the same orbit. Over vegetation: 
- **Before mowing**: tall, moving grass causes strong temporal decorrelation --> low coherence expected (~0.2–0.4)
- **After mowing**: short, stable stubble shows much higher coherence expected (~0.5–0.8)
- **coherence_jump = coherence_after − coherence_before** is expected to be the primary mowing signal

[Tamm et. al](https://www.mdpi.com/2072-4292/8/10/802) used coherence as mowing signal with better success. Therefore their approach is replicated in this and the following notebook. 

This notebook acquires the SLC scenes needed to compute coherence.

### Temporal logic: SLC coherence quadruplet

For each mowing event **four SLC acquisitions** from the **same relative orbit** are needed, following the **Quadruplet** strategy per Tamm et al. (2016):

- coherence_before = coh(SLC_t1, SLC_t2)   --> low if grass is growing (both acquisitions before mowing)
- coherence_after  = coh(SLC_t3, SLC_t4)   --> high if grass was recently cut (both after mowing)
- coherence_jump   = coherence_after − coherence_before   <-- expected to be key feature

**Why quadruplet instead of triplet?** A naive triplet (t1, t2, t3) would compute coh_after = coh(t2, t3), where t2 is *before* mowing and t3 is *after*. This cross-event pair straddles the mowing event and cannot measure post-mowing surface stability — the decorrelation from the cut itself dominates. The quadruplet places **both** after-pair scenes post-mowing, so coh(t3, t4) measures how stable the stubble surface is, giving a much cleaner mowing signal.

**Critical constraint:** both scenes in a coherence pair must be from the **same relative orbit** and same subswath. Mixing orbits destroys phase coherence.

### Disk space management

One S1 IW SLC product ≈ 8 GB. With over a hundred unique scenes needed for the computation, this means disk space is an issue. Therefore acquisition and processing are handeled seperatly and in batches.

This notebook handles only steps 1–5 (query, plan, and batch download). Notebook 06 does the SNAP processing. After processing a batch, raw SLC files are deleted.

### Notebook workflow

1. Define AOI (from S2 reference tile, same as nb02)
2. Configure CDSE credentials
3. Query the full S1 IW SLC catalogue over the AOI (metadata only, no download)
4. For each event, find the best same-orbit SLC quadruplet (t1, t2, t3, t4)
5. Build `data/slc_scene_index.csv` and `data/slc_event_coverage.csv`
6. Batch download of SLC scenes (with skip logic for already-downloaded files)
7. SAFE file management: identify files needed, on disk, and safe to delete

---
## 1. Setup

In [136]:
import os
import glob
import json
import configparser
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
from datetime import timedelta

from cdsetool.credentials import Credentials
from cdsetool.query import query_features, describe_collection
from cdsetool.download import download_features
from dotenv import load_dotenv
load_dotenv()

# Paths 
S2_DIR = r'..\data\Sentinel_CH'
TEMPORAL_MATCHES = r'..\data\temporal_matches.csv'
SLC_DIR = r'..\data\Sentinel_S1_SLC'
SLC_INDEX_PATH = r'..\data\slc_scene_index.csv'
SLC_COVERAGE_PATH = r'..\data\slc_event_coverage.csv'
COH_DIR = r'..\data\features_coherence'

os.makedirs(SLC_DIR, exist_ok=True)

# SNAP configuration
SNAP_HOME = r'C:\Program Files\esa-snap'
_pyrosar_cfg = os.path.join(os.path.expanduser('~'), '.pyrosar', 'config.ini')
if not os.path.exists(_pyrosar_cfg):
    os.makedirs(os.path.dirname(_pyrosar_cfg), exist_ok=True)
    _cfg = configparser.RawConfigParser()
    _cfg.add_section('SNAP')
    _cfg.set('SNAP', 'path', os.path.join(SNAP_HOME, 'bin', 'snap64.exe'))
    _cfg.set('SNAP', 'gpt', os.path.join(SNAP_HOME, 'bin', 'gpt.exe'))
    _cfg.set('SNAP', 'etc', os.path.join(SNAP_HOME, 'etc'))
    with open(_pyrosar_cfg, 'w') as f:
        _cfg.write(f)
    print(f'pyroSAR config written: {_pyrosar_cfg}')
else:
    print(f'pyroSAR config exists:  {_pyrosar_cfg}')

# Parameters 
# S1 repeat cycle at Zürich: 12 days (Sentinel-1A only post-2022; 6 days with A+B)
# We accept 6 or 12 day baselines.
VALID_BASELINES_DAYS = [6, 12]
# Search window around each S2 date when looking for a matching SLC
# SLC triplet dates won't coincide exactly with S2 dates, we search within ±8 days
SLC_MATCH_TOLERANCE = 8

print('Setup complete.')

pyroSAR config exists:  C:\Users\flori\.pyrosar\config.ini
Setup complete.


In [137]:
# known bad events to skip in analysis (from nb06 processing log)
#SKIP_MATCH_IDS = {18, 19, 20, 24, 25, 26, 27, 31, 32, 36, 37, 38, 40, 54, 55, 56, 57, 60, 61, 68, 86, 87, 88, 89, 90, 91}  

In [150]:
SKIP_MATCH_IDS = {15, 16, 17, 18, 19, 20, 22, 24, 25, 26, 27, 31, 32, 34, 35, 36, 37, 38, 40, 52, 53, 54, 55, 56, 57, 60, 61, 67, 68, 79, 86, 87, 88, 89, 90, 91}   

---
## 2. CDSE Credentials

In [139]:
CDSE_USERNAME = os.environ.get('CDSE_USERNAME', '')  
CDSE_PASSWORD = os.environ.get('CDSE_PASSWORD', '')  

if not CDSE_USERNAME or not CDSE_PASSWORD:
    raise ValueError(
        'CDSE credentials not set. '
        'Set environment variables CDSE_USERNAME and CDSE_PASSWORD'
    )

credentials = Credentials(username=CDSE_USERNAME, password=CDSE_PASSWORD)
# Verify credentials by opening a session
session = credentials.get_session()
print(f'CDSE credentials verified. Session: {type(session).__name__}')

CDSE credentials verified. Session: Session


---
## 3. Define Area of Interest

Same AOI derivation as notebook 03, from the S2 reference tile bounding box.

In [140]:
s2_files = sorted(glob.glob(os.path.join(S2_DIR, '*.tif')))
if not s2_files:
    raise FileNotFoundError(f'No S2 GeoTIFFs found in {S2_DIR}')

with rasterio.open(s2_files[0]) as s2:
    b = s2.bounds  # EPSG:2056

transformer = Transformer.from_crs('EPSG:2056', 'EPSG:4326', always_xy=True)
lon_min, lat_min = transformer.transform(b.left,  b.bottom)
lon_max, lat_max = transformer.transform(b.right, b.top)

# WKT polygon for CDSE query
aoi_wkt = (
    f'POLYGON(('
    f'{lon_min:.5f} {lat_min:.5f}, '
    f'{lon_max:.5f} {lat_min:.5f}, '
    f'{lon_max:.5f} {lat_max:.5f}, '
    f'{lon_min:.5f} {lat_max:.5f}, '
    f'{lon_min:.5f} {lat_min:.5f}'
    f'))'
)

print(f'S2 reference tile: {os.path.basename(s2_files[0])}')
print(f'AOI (EPSG:2056):   X=[{b.left:.1f}, {b.right:.1f}]  Y=[{b.bottom:.1f}, {b.top:.1f}]')
print(f'AOI (WGS84):       lon=[{lon_min:.5f}, {lon_max:.5f}]  lat=[{lat_min:.5f}, {lat_max:.5f}]')
print(f'AOI WKT:           {aoi_wkt}')

S2 reference tile: 2019-03-01.tif
AOI (EPSG:2056):   X=[2682012.9, 2685824.6]  Y=[1254788.2, 1260380.7]
AOI (WGS84):       lon=[8.52587, 8.57747]  lat=[47.43878, 47.48859]
AOI WKT:           POLYGON((8.52587 47.43878, 8.57747 47.43878, 8.57747 47.48859, 8.52587 47.48859, 8.52587 47.43878))


---
## 4. Query CDSE Catalogue (metadata only)

Fetch all S1 IW SLC ascending products over the study area from 2019–2023. This is a metadata-only query, no data is downloaded yet.

The entire date range is queried once and the results are cached locally to avoid repeated API calls during the pairing logic.

In [141]:
CATALOGUE_CACHE = r'..\data\slc_catalogue_cache.json'

if os.path.exists(CATALOGUE_CACHE):
    print('Loading catalogue from cache...')
    with open(CATALOGUE_CACHE, 'r') as f:
        catalogue_raw = json.load(f)
else:
    print('Querying CDSE catalogue (ascending IW SLC, 2019-01-01 to 2023-12-31)...')
    print('This may take 1-2 minutes.')
    # query_features uses the public OData API
    features = list(query_features(
        'SENTINEL-1',
        {
            'contentDateStartGe': '2019-01-01',
            'contentDateEndLe':   '2023-12-31',
            'productType':        'IW_SLC__1S',
            'operationalMode':    'IW',
            'orbitDirection':     'ASCENDING',
            'geometry':           aoi_wkt,
        },
    ))
    catalogue_raw = features
    with open(CATALOGUE_CACHE, 'w') as f:
        json.dump(catalogue_raw, f)
    print(f'Saved {len(catalogue_raw)} results to cache.')

print(f'\nTotal S1 IW SLC ascending products over AOI (2019-2023): {len(catalogue_raw)}')
if catalogue_raw:
    sample = catalogue_raw[0]
    # OData API returns flat dict with capitalized keys (Id, Name, ContentDate, ...)
    print(f'Sample product ID: {sample["Id"]}')
    print(f'Sample name:       {sample["Name"]}')

Loading catalogue from cache...

Total S1 IW SLC ascending products over AOI (2019-2023): 633
Sample product ID: b014c03b-cdef-5ce9-85b3-ce6f72585f23
Sample name:       S1A_IW_SLC__1SDV_20190101T171515_20190101T171542_025287_02CC09_0A0B.SAFE


In [142]:
# Parse catalogue into a structured DataFrame
# OData response format: flat dict with keys Id, Name, ContentDate, GeoFootprint, ...
# Name format: S1A_IW_SLC__1SDV_20190101T171515_20190101T171542_025287_02CC09_XXXX.SAFE
# Split by '_': [0]=S1A [1]=IW [2]=SLC [3]='' [4]=1SDV [5]=date1 [6]=date2 [7]=abs_orbit
#                                       ^ empty string from double-underscore SLC__1SDV

def _rel_orbit(abs_orbit: int, platform: str) -> int:
    """Convert S1 absolute orbit to relative orbit number."""
    if 'S1A' in platform:
        return (abs_orbit - 73) % 175 + 1
    elif 'S1B' in platform:
        return (abs_orbit - 27) % 175 + 1
    return -1

# Parse catalogue into a structured DataFrame
records = []
for feat in catalogue_raw:
    product_id = feat['Id']
    name       = feat['Name'].replace('.SAFE', '')  # strip extension

    # Date from ContentDate.Start (ISO-8601)
    start_str  = feat.get('ContentDate', {}).get('Start', '')

    # Parse name components
    parts    = name.split('_')
    platform = parts[0] if len(parts) > 0 else ''       # S1A / S1B
    try:
        abs_orbit = int(parts[7])                        # index 7 due to double-underscore in SLC__1SDV
    except (IndexError, ValueError):
        abs_orbit = -1

    orbit_rel = _rel_orbit(abs_orbit, platform) if abs_orbit > 0 else -1

    records.append({
        'product_id': product_id,
        'name':       name + '.SAFE',                    # keep .SAFE suffix for file matching
        'date':       pd.to_datetime(start_str[:10]) if start_str else pd.NaT,
        'orbit_rel':  orbit_rel,
        'orbit_abs':  abs_orbit,
        'platform':   platform,
    })
# Create DataFrame, drop rows with missing dates, sort by date, and normalize to midnight
slc_catalogue = (
    pd.DataFrame(records)
    .dropna(subset=['date'])
    .sort_values('date')
    .reset_index(drop=True)
)
slc_catalogue['date'] = pd.to_datetime(slc_catalogue['date']).dt.normalize()

# Verification and summary prints
print(f'Parsed catalogue: {len(slc_catalogue)} scenes')
print(f'Date range: {slc_catalogue["date"].min().date()} to {slc_catalogue["date"].max().date()}')
print(f'Unique relative orbits: {sorted(slc_catalogue["orbit_rel"].unique())}')
print(f'Unique platforms: {sorted(slc_catalogue["platform"].unique())}')
print('\nSample rows:')
print(slc_catalogue.head(8).to_string(index=False))

Parsed catalogue: 633 scenes
Date range: 2019-01-01 to 2023-12-30
Unique relative orbits: [np.int64(15), np.int64(88)]
Unique platforms: ['S1A', 'S1B']

Sample rows:
                          product_id                                                                     name       date  orbit_rel  orbit_abs platform
b014c03b-cdef-5ce9-85b3-ce6f72585f23 S1A_IW_SLC__1SDV_20190101T171515_20190101T171542_025287_02CC09_0A0B.SAFE 2019-01-01         15      25287      S1A
13a35855-34d7-52e4-8630-9d5e9e5e244c S1A_IW_SLC__1SDV_20190101T171539_20190101T171606_025287_02CC09_C6BF.SAFE 2019-01-01         15      25287      S1A
0f917fb3-9f1a-5576-8ddb-ed0b817731a3 S1A_IW_SLC__1SDV_20190106T172345_20190106T172412_025360_02CEAE_E0E9.SAFE 2019-01-06         88      25360      S1A
c9f52f65-38bc-58cb-aa94-294ee5b79cf2 S1B_IW_SLC__1SDV_20190107T171445_20190107T171512_014391_01AC8B_AFEA.SAFE 2019-01-07         15      14391      S1B
aeeab18a-bf11-5a00-8bb1-e08c46002713 S1B_IW_SLC__1SDV_20190112T172256_2019

---
## 5. Determine Target Relative Orbit

For coherence to work, all scenes in a pair must share the **same relative orbit number**.
The cover of each relative orbit over the study area is inspected and the one with the best temporal coverage over 2019–2023 is chosen.


In [143]:
# Compute orbit statistics
orbit_stats = (
    slc_catalogue
    .groupby('orbit_rel')
    .agg(
        n_scenes=('date', 'count'),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
    )
    .sort_values('n_scenes', ascending=False)
)

print('=== Ascending orbit tracks over study area ===')
print(orbit_stats.to_string())

# Pick the orbit with the most scenes (best temporal sampling)
TARGET_ORBIT = int(orbit_stats.index[0])
print(f'\nSelected relative orbit: {TARGET_ORBIT}  ({orbit_stats.loc[TARGET_ORBIT, "n_scenes"]} scenes)')

# Filter catalogue to selected orbit
orbit_catalogue = slc_catalogue[slc_catalogue['orbit_rel'] == TARGET_ORBIT].copy().reset_index(drop=True)

# Compute temporal baselines between consecutive acquisitions
orbit_catalogue['baseline_to_prev'] = orbit_catalogue['date'].diff().dt.days

print(f'\nBaseline distribution for orbit {TARGET_ORBIT}:')
print(orbit_catalogue['baseline_to_prev'].dropna().value_counts().sort_index())

=== Ascending orbit tracks over study area ===
           n_scenes first_date  last_date
orbit_rel                                
15              387 2019-01-01 2023-12-30
88              246 2019-01-06 2023-12-23

Selected relative orbit: 15  (387 scenes)

Baseline distribution for orbit 15:
baseline_to_prev
0.0     154
6.0     160
12.0     72
Name: count, dtype: int64


---
## 6. Match SLC Quadruplets to Mowing Events

For each mowing event, find four SLC acquisitions (from the target orbit) that:
1. Bracket the S2 `before_far` date → gives SLC_t1 (earliest before scene)
2. Bracket the S2 `before_near` date → gives SLC_t2 (latest before scene)
3. Bracket the S2 `after` date, strictly after event → gives SLC_t3 (first post-event scene)
4. Follow t3 by exactly 6 or 12 days, still after event → gives SLC_t4 (second post-event scene)

**Quadruplet design (Tamm et al. 2016):**
- `coherence_before = coh(t1, t2)` — both before mowing → measures pre-mow surface stability
- `coherence_after  = coh(t3, t4)` — both after mowing → measures post-mow stubble stability
- Both t3 and t4 must be strictly after the event date

Events where no valid t4 exists are flagged (`valid_after_quad = False`) and will be skipped in NB06. The old `baseline_after` column (t3 − t2) is retained for reference.

Matching strategy:
- Find all SLC acquisitions within `±SLC_MATCH_TOLERANCE` days of each S2 triplet date
- Prefer the acquisition closest to the S2 date
- Verify temporal boundary constraints (t1, t2 before event; t3, t4 after event)
- Verify that consecutive SLC pairs have a valid temporal baseline (6 or 12 days)
- For t4: search the orbit catalogue for an acquisition exactly 6 or 12 days after t3

In [144]:
# Build SLC quadruplet index for each temporal match
matches_df = pd.read_csv(TEMPORAL_MATCHES)
matches_df['event_date'] = pd.to_datetime(matches_df['event_date'])
matches_df['before_far_date'] = pd.to_datetime(matches_df['before_far_file'].str.replace('.tif', '', regex=False))
matches_df['before_near_date'] = pd.to_datetime(matches_df['before_near_file'].str.replace('.tif', '', regex=False))
matches_df['after_date'] = pd.to_datetime(matches_df['after_file'].str.replace('.tif', '', regex=False))
matches_df['match_id'] = matches_df.index

orbit_dates = orbit_catalogue['date'].values  # numpy array of datetime64
orbit_index = orbit_catalogue.set_index('date')

def find_nearest_slc(target_date, tolerance_days=SLC_MATCH_TOLERANCE):
    """Return the orbit_catalogue row whose date is closest to target_date within tolerance."""
    target = pd.Timestamp(target_date)
    window = orbit_catalogue[
        (orbit_catalogue['date'] >= target - timedelta(days=tolerance_days)) &
        (orbit_catalogue['date'] <= target + timedelta(days=tolerance_days))
    ]
    if len(window) == 0:
        return None
    idx = (window['date'] - target).abs().idxmin()
    return window.loc[idx]


def find_all_slc_within_tolerance(target_date, tolerance_days=SLC_MATCH_TOLERANCE):
    """Return all SLC acquisitions within tolerance window, sorted by distance to target."""
    target = pd.Timestamp(target_date)
    window = orbit_catalogue[
        (orbit_catalogue['date'] >= target - timedelta(days=tolerance_days)) &
        (orbit_catalogue['date'] <= target + timedelta(days=tolerance_days))
    ].copy()
    window['distance'] = (window['date'] - target).abs()
    return window.sort_values('distance')


def check_valid_baseline(date1, date2):
    """Check that the temporal baseline between two SLC acquisitions is 6 or 12 days."""
    if date1 is None or date2 is None:
        return False
    delta = abs((pd.Timestamp(date2) - pd.Timestamp(date1)).days)
    return delta in VALID_BASELINES_DAYS


# Build the SLC quadruplet index with temporal boundary validation
slc_index_records = []
boundary_corrections = []
n_corrected = 0
n_invalidated = 0

for _, row in matches_df.iterrows():
    match_id = int(row['match_id'])
    event_date = pd.Timestamp(row['event_date'])

    slc_t1 = find_nearest_slc(row['before_far_date'])
    slc_t2 = find_nearest_slc(row['before_near_date'])
    slc_t3 = find_nearest_slc(row['after_date'])

    # Fallback: if t2 and t3 are from the same date, find the next acquisition for t3
    if slc_t2 is not None and slc_t3 is not None:
        if slc_t2['date'] == slc_t3['date']:
            further = orbit_catalogue[
                orbit_catalogue['date'] > slc_t2['date'] + timedelta(days=1)
            ]
            if len(further) > 0:
                slc_t3 = further.iloc[0]

    # Apply temporal boundary constraints
    # Before-pair: both t1 and t2 must be strictly before event_date
    if slc_t1 is not None and pd.Timestamp(slc_t1['date']) >= event_date:
        candidates_t1 = find_all_slc_within_tolerance(row['before_far_date'])
        found_valid = False
        for _, cand in candidates_t1.iterrows():
            if pd.Timestamp(cand['date']) < event_date:
                slc_t1_old = slc_t1['date']
                slc_t1 = cand
                boundary_corrections.append(f"  match_id {match_id}: t1 {slc_t1_old} → {slc_t1['date']}")
                found_valid = True
                n_corrected += 1
                break
        if not found_valid:
            slc_t1 = None

    if slc_t2 is not None and pd.Timestamp(slc_t2['date']) >= event_date:
        candidates_t2 = find_all_slc_within_tolerance(row['before_near_date'])
        found_valid = False
        for _, cand in candidates_t2.iterrows():
            if pd.Timestamp(cand['date']) < event_date:
                slc_t2_old = slc_t2['date']
                slc_t2 = cand
                boundary_corrections.append(f"  match_id {match_id}: t2 {slc_t2_old} → {slc_t2['date']}")
                found_valid = True
                n_corrected += 1
                break
        if not found_valid:
            slc_t2 = None

    # After-pair: t3 must be strictly after event_date; t2 must be strictly before
    if slc_t2 is not None and pd.Timestamp(slc_t2['date']) >= event_date:
        slc_t2 = None

    if slc_t3 is not None and pd.Timestamp(slc_t3['date']) <= event_date:
        candidates_t3 = find_all_slc_within_tolerance(row['after_date'])
        found_valid = False
        for _, cand in candidates_t3.iterrows():
            if pd.Timestamp(cand['date']) > event_date:
                slc_t3_old = slc_t3['date']
                slc_t3 = cand
                boundary_corrections.append(f"  match_id {match_id}: t3 {slc_t3_old} → {slc_t3['date']}")
                found_valid = True
                n_corrected += 1
                break
        if not found_valid:
            slc_t3 = None

    t1_date = slc_t1['date'] if slc_t1 is not None else pd.NaT
    t2_date = slc_t2['date'] if slc_t2 is not None else pd.NaT
    t3_date = slc_t3['date'] if slc_t3 is not None else pd.NaT

    t1_id = slc_t1['product_id'] if slc_t1 is not None else None
    t2_id = slc_t2['product_id'] if slc_t2 is not None else None
    t3_id = slc_t3['product_id'] if slc_t3 is not None else None

    t1_name = slc_t1['name'] if slc_t1 is not None else None
    t2_name = slc_t2['name'] if slc_t2 is not None else None
    t3_name = slc_t3['name'] if slc_t3 is not None else None

    # Compute baselines and validate
    baseline_before = abs((t2_date - t1_date).days) if pd.notna(t1_date) and pd.notna(t2_date) else None
    baseline_after = abs((t3_date - t2_date).days) if pd.notna(t2_date) and pd.notna(t3_date) else None

    valid_before = baseline_before in VALID_BASELINES_DAYS if baseline_before is not None else False
    valid_after = baseline_after  in VALID_BASELINES_DAYS if baseline_after  is not None else False
    full_ok = valid_before and valid_after

    # Final boundary check (after all corrections)
    if full_ok:
        t1_before_event = pd.Timestamp(t1_date) < event_date
        t2_before_event = pd.Timestamp(t2_date) < event_date
        t3_after_event = pd.Timestamp(t3_date) > event_date
        if not (t1_before_event and t2_before_event and t3_after_event):
            full_ok = False
            n_invalidated += 1

    # --- Quadruplet: find t4 ---
    # t4 must be exactly 6 or 12 days after t3, strictly after event_date.
    # Both t3 and t4 post-event ensures coh(t3,t4) measures post-mow stability, not the mowing event itself.
    t4_date = pd.NaT
    t4_id = None
    t4_name = None
    baseline_after_quad = None
    valid_after_quad = False

    if full_ok and slc_t3 is not None and pd.notna(t3_date):
        t3_dt = pd.Timestamp(t3_date)
        for delta in sorted(VALID_BASELINES_DAYS):
            t4_target = t3_dt + timedelta(days=delta)
            cands = orbit_catalogue[orbit_catalogue['date'] == t4_target]
            if len(cands) > 0:
                candidate = cands.iloc[0]
                if pd.Timestamp(candidate['date']) > event_date:
                    t4_date = candidate['date']
                    t4_id = candidate['product_id']
                    t4_name = candidate['name']
                    baseline_after_quad = delta
                    valid_after_quad = True
                    break

    # Append record to index (keep all old columns for reference, add t4 columns)
    slc_index_records.append({
        'match_id': match_id,
        'event_date_str': row['event_date_str'],
        's2_before_far': row['before_far_date'].date(),
        's2_before_near': row['before_near_date'].date(),
        's2_after': row['after_date'].date(),
        'slc_t1_date': t1_date.date() if pd.notna(t1_date) else None,
        'slc_t2_date': t2_date.date() if pd.notna(t2_date) else None,
        'slc_t3_date': t3_date.date() if pd.notna(t3_date) else None,
        'slc_t4_date': t4_date.date() if pd.notna(t4_date) else None,
        'slc_t1_id': t1_id,
        'slc_t2_id': t2_id,
        'slc_t3_id': t3_id,
        'slc_t4_id': t4_id,
        'slc_t1_name': t1_name,
        'slc_t2_name': t2_name,
        'slc_t3_name': t3_name,
        'slc_t4_name': t4_name,
        'baseline_before': baseline_before,
        'baseline_after': baseline_after,          # old t2→t3, kept for reference
        'baseline_after_quad': baseline_after_quad, # new t3→t4
        'valid_before_pair': valid_before,
        'valid_after_pair': valid_after,
        'full_triplet_ok': full_ok,
        'valid_after_quad': valid_after_quad,
    })

# Create DataFrame and print summary
slc_index_df = pd.DataFrame(slc_index_records)
n_clean = slc_index_df['full_triplet_ok'].sum()
n_full  = slc_index_df['full_triplet_ok'].sum()
n_total = len(slc_index_df)

print(f'=== SLC Quadruplet Matching Summary ===')
print(f'Events with full valid SLC triplet (t1/t2/t3):  {n_full} / {n_total}  ({n_full/n_total*100:.0f}%)')
print(f'Events with valid before pair only:              {slc_index_df["valid_before_pair"].sum()}')
print(f'Events with valid after pair only:               {slc_index_df["valid_after_pair"].sum()}')

print('\nBaseline distribution (before pair t1→t2):')
print(slc_index_df['baseline_before'].value_counts().sort_index())
print('\nBaseline distribution (old after pair t2→t3, reference only):')
print(slc_index_df['baseline_after'].value_counts().sort_index())

# Boundary correction summary
print(f'\n=== Temporal Boundary Validation ===')
print(f"Events with boundary-corrected matches: {n_corrected}")
print(f"Events invalidated by boundary constraint: {n_invalidated}")
print(f"Clean valid triplets (both pairs respect event boundary): {n_clean}")

if boundary_corrections:
    print(f'\nBoundary corrections applied:')
    for msg in boundary_corrections[:10]:
        print(msg)
    if len(boundary_corrections) > 10:
        print(f'  ... and {len(boundary_corrections) - 10} more')

# Quadruplet summary
n_quad = slc_index_df[slc_index_df['full_triplet_ok'] & slc_index_df['valid_after_quad']].shape[0]
n_fallback = n_full - n_quad
print(f'\n=== Quadruplet Matching Results ===')
print(f'Events with valid t4 (full quadruplet): {n_quad} / {n_full}')
print(f'Events without valid t4 (will skip in NB06): {n_fallback}')
print(f'\nBaseline distribution (new after pair t3→t4):')
print(slc_index_df.loc[slc_index_df['valid_after_quad'], 'baseline_after_quad'].value_counts().sort_index())

# Show events that failed triplet matching
failed = slc_index_df[~slc_index_df['full_triplet_ok']]
if len(failed) > 0:
    print(f'\n=== {len(failed)} events WITHOUT a valid SLC triplet ===')
    cols = ['event_date_str', 'baseline_before', 'baseline_after', 'valid_before_pair', 'valid_after_pair']
    print(failed[cols].to_string(index=False))

=== SLC Quadruplet Matching Summary ===
Events with full valid SLC triplet (t1/t2/t3):  72 / 92  (78%)
Events with valid before pair only:              73
Events with valid after pair only:               84

Baseline distribution (before pair t1→t2):
baseline_before
0.0     18
6.0     46
12.0    27
Name: count, dtype: int64

Baseline distribution (old after pair t2→t3, reference only):
baseline_after
6.0     40
12.0    44
18.0     3
24.0     3
Name: count, dtype: int64

=== Temporal Boundary Validation ===
Events with boundary-corrected matches: 19
Events invalidated by boundary constraint: 0
Clean valid triplets (both pairs respect event boundary): 72

Boundary corrections applied:
  match_id 4: t3 2019-05-31 00:00:00 → 2019-06-06 00:00:00
  match_id 6: t3 2019-06-06 00:00:00 → 2019-06-12 00:00:00
  match_id 20: t3 2019-07-18 00:00:00 → 2019-07-24 00:00:00
  match_id 22: t3 2019-07-24 00:00:00 → 2019-07-30 00:00:00
  match_id 23: t3 2019-07-24 00:00:00 → 2019-07-30 00:00:00
  match_id

---
## 7. Build Scene Index and Coverage Table

Save the full quadruplet index and a simplified coverage table for use in notebook 06. New columns vs the old triplet index: `slc_t4_date`, `slc_t4_id`, `slc_t4_name`, `baseline_after_quad` (t4 − t3 in days), `valid_after_quad` (bool).
The old `baseline_after` (t3 − t2) is kept for reference.

In [145]:
# Unique SLC scenes needed (de-duplicate; many events share t2)
# Include t4 scenes for quadruplet-valid events
valid_triplet = slc_index_df[slc_index_df['full_triplet_ok']]
valid_quad    = slc_index_df[slc_index_df['full_triplet_ok'] & slc_index_df['valid_after_quad']]

t1_scenes = valid_triplet[['slc_t1_id', 'slc_t1_name', 'slc_t1_date']].rename(
    columns={'slc_t1_id': 'product_id', 'slc_t1_name': 'name', 'slc_t1_date': 'date'})
t2_scenes = valid_triplet[['slc_t2_id', 'slc_t2_name', 'slc_t2_date']].rename(
    columns={'slc_t2_id': 'product_id', 'slc_t2_name': 'name', 'slc_t2_date': 'date'})
t3_scenes = valid_triplet[['slc_t3_id', 'slc_t3_name', 'slc_t3_date']].rename(
    columns={'slc_t3_id': 'product_id', 'slc_t3_name': 'name', 'slc_t3_date': 'date'})
t4_scenes = valid_quad[['slc_t4_id', 'slc_t4_name', 'slc_t4_date']].rename(
    columns={'slc_t4_id': 'product_id', 'slc_t4_name': 'name', 'slc_t4_date': 'date'})

all_ids = (
    pd.concat([t1_scenes, t2_scenes, t3_scenes, t4_scenes])
    .dropna(subset=['product_id'])
    .drop_duplicates(subset='product_id')
    .sort_values('date')
    .reset_index(drop=True)
)

# How many t4 scenes are new (not already needed by t1/t2/t3)?
t123_ids = (
    pd.concat([t1_scenes, t2_scenes, t3_scenes])
    .dropna(subset=['product_id'])
    .drop_duplicates(subset='product_id')
)
t4_only_new = all_ids[~all_ids['product_id'].isin(t123_ids['product_id'])]

print(f'Unique SLC scenes to download (t1+t2+t3+t4): {len(all_ids)}')
print(f'  Of which t4-only new scenes:               {len(t4_only_new)}')
print(f'  (From {n_full} triplet-valid events; {valid_quad.shape[0]} have a valid t4)')
print(f'Estimated disk space: {len(all_ids) * 8:.0f} GB (at ~8 GB per scene)')

# Save full index
slc_index_df.to_csv(SLC_INDEX_PATH, index=False)
print(f'\nSaved: {SLC_INDEX_PATH}')

# Simplified coverage table
coverage_df = slc_index_df[[
    'match_id', 'event_date_str',
    'slc_t1_date', 'slc_t2_date', 'slc_t3_date', 'slc_t4_date',
    'slc_t1_id',  'slc_t2_id',  'slc_t3_id',  'slc_t4_id',
    'baseline_before', 'baseline_after', 'baseline_after_quad',
    'valid_before_pair', 'valid_after_pair', 'full_triplet_ok', 'valid_after_quad',
]].copy()
coverage_df.to_csv(SLC_COVERAGE_PATH, index=False)
print(f'Saved: {SLC_COVERAGE_PATH}')

# Quadruplet download summary
print(f'\n=== Quadruplet Matching Results ===')
print(f'  Events with valid t4:             {valid_quad.shape[0]} / {n_full}')
print(f'  Events without valid t4 (skip):   {n_full - valid_quad.shape[0]}')
print(f'  New unique SLC scenes needed (t4 only): {len(t4_only_new)}')
print(f'  Total unique SLC scenes to download:    {len(all_ids)}')

# Print one example event showing the full quadruplet
example_rows = slc_index_df[slc_index_df['full_triplet_ok'] & slc_index_df['valid_after_quad']]
if len(example_rows) > 0:
    ex = example_rows.iloc[0]
    print(f'\n=== Example full quadruplet (match_id={ex["match_id"]}, event={ex["event_date_str"]}) ===')
    print(f'  t1: {ex["slc_t1_date"]}  (before-pair master)')
    print(f'  t2: {ex["slc_t2_date"]}  (before-pair slave,  baseline_before={ex["baseline_before"]}d)')
    print(f'  t3: {ex["slc_t3_date"]}  (after-pair master)')
    print(f'  t4: {ex["slc_t4_date"]}  (after-pair slave,   baseline_after_quad={ex["baseline_after_quad"]}d)')

Unique SLC scenes to download (t1+t2+t3+t4): 74
  Of which t4-only new scenes:               7
  (From 72 triplet-valid events; 72 have a valid t4)
Estimated disk space: 592 GB (at ~8 GB per scene)

Saved: ..\data\slc_scene_index.csv
Saved: ..\data\slc_event_coverage.csv

=== Quadruplet Matching Results ===
  Events with valid t4:             72 / 72
  Events without valid t4 (skip):   0
  New unique SLC scenes needed (t4 only): 7
  Total unique SLC scenes to download:    74

=== Example full quadruplet (match_id=0, event=20190509) ===
  t1: 2019-05-01  (before-pair master)
  t2: 2019-05-07  (before-pair slave,  baseline_before=6.0d)
  t3: 2019-05-13  (after-pair master)
  t4: 2019-05-19  (after-pair slave,   baseline_after_quad=6.0d)


---
## 8. Download SLC Scenes (Batch Mode)

**Disk space management:**
- `BATCH_SIZE` controls how many scenes to download per run.
- After each batch, notebook 06 is ran on this batch first to process the downloaded scenes, then the raw `.SAFE` directories are deleted from `data/Sentinel_S1_SLC/` to make space for the next batch.
- Scenes already on disk are automatically skipped.


In [146]:
def slc_safe_name(product_name):
    """Convert product name to expected .SAFE directory name."""
    if not product_name:
        return None
    name = product_name.replace('.zip', '').replace('.SAFE', '')
    return name + '.SAFE'


def is_on_disk(product_name, slc_dir=SLC_DIR):
    """Check if an SLC scene is already downloaded (.SAFE dir, .zip, or .SAFE.zip)."""
    if not product_name:
        return False
    safe_name = slc_safe_name(product_name)          # e.g. S1A_...SAFE
    return (
        os.path.isdir( os.path.join(slc_dir, safe_name))           or  # extracted
        os.path.isfile(os.path.join(slc_dir, safe_name + '.zip'))   or  # CDSE download
        os.path.isfile(os.path.join(slc_dir, safe_name.replace('.SAFE', '.zip')))  # plain zip
    )

# Check if coherence products for a given match_id are already on disk
def is_coherence_done(match_id, coh_dir=COH_DIR):
    """Return True if both coherence TIFs for this event are already on disk."""
    before = os.path.join(coh_dir, f'{int(match_id)}_coh_before.tif')
    after  = os.path.join(coh_dir, f'{int(match_id)}_coh_after.tif')
    return os.path.isfile(before) and os.path.isfile(after)


# Download configuration 
PILOT_MODE = False    # <- set False once pilot passes
BATCH_SIZE = 15

# Build download list with blacklist skip logic
to_download_all = []
skipped_safe_files = []
n_scenes_processed = 0

for _, row in all_ids.iterrows():
    product_id = row['product_id']
    product_name = row['name']
    n_scenes_processed += 1
    
    # Skip if already on disk
    if is_on_disk(product_name):
        continue
    
    # Find all events that need this SAFE file (t1, t2, t3, or t4)
    event_refs = slc_index_df[
        slc_index_df['full_triplet_ok'] & (
            (slc_index_df['slc_t1_id'] == product_id) |
            (slc_index_df['slc_t2_id'] == product_id) |
            (slc_index_df['slc_t3_id'] == product_id) |
            (slc_index_df['slc_t4_id'] == product_id)
        )
    ]
    
    # Get set of match IDs that reference this file
    ref_match_ids = set(int(mid) for mid in event_refs['match_id'])
    
    # If ALL referencing events are blacklisted, skip download
    if ref_match_ids and ref_match_ids.issubset(SKIP_MATCH_IDS):
        skipped_safe_files.append(product_name)
        continue
    
    # Check if at least one non-blacklisted event still needs this file
    needs_file = False
    for mid in ref_match_ids:
        if int(mid) not in SKIP_MATCH_IDS and not is_coherence_done(mid):
            needs_file = True
            break
    
    if needs_file:
        to_download_all.append(row.to_dict())

already_done = len(all_ids) - len(to_download_all) - len(skipped_safe_files)

# In pilot mode, download the first quadruplet-valid event's four scenes
if PILOT_MODE:
    first_event = slc_index_df[
        slc_index_df['full_triplet_ok'] & slc_index_df['valid_after_quad']
    ].iloc[0]
    pilot_ids = {
        first_event['slc_t1_id'], first_event['slc_t2_id'],
        first_event['slc_t3_id'], first_event['slc_t4_id'],
    }
    to_download = [r for r in to_download_all if r['product_id'] in pilot_ids]
    print(f'PILOT MODE: downloading {len(to_download)} scenes for event '
          f'{first_event["event_date_str"]}  (match_id={int(first_event["match_id"])})')
    print(f'  t1: {first_event["slc_t1_name"]}')
    print(f'  t2: {first_event["slc_t2_name"]}')
    print(f'  t3: {first_event["slc_t3_name"]}')
    print(f'  t4: {first_event["slc_t4_name"]}')
else:
    to_download = to_download_all[:BATCH_SIZE]

print(f'Total unique scenes:         {len(all_ids)}')
print(f'Already processed (skipped): {already_done}')
print(f'Blacklist-only (skipped):    {len(skipped_safe_files)}')
if skipped_safe_files:
    print(f'  Skipped downloading {len(skipped_safe_files)} SAFE file(s) (only needed by blacklisted events)')
print(f'This run will download:      {len(to_download)}')
print(f'Estimated size:              ~{len(to_download) * 7}-{len(to_download) * 8} GB  (~7-8 GB/scene)')

Total unique scenes:         74
Already processed (skipped): 50
Blacklist-only (skipped):    13
  Skipped downloading 13 SAFE file(s) (only needed by blacklisted events)
This run will download:      11
Estimated size:              ~77-88 GB  (~7-8 GB/scene)


In [147]:
# Download the current batch
batch = to_download[:BATCH_SIZE] if not PILOT_MODE else to_download

if not batch:
    print('All scenes already on disk. Nothing to download.')
else:
    print(f'Starting download of {len(batch)} scenes to {SLC_DIR}...')
    print('This will take several hours. Monitor disk space during download.')
    print()

    # Build list of feature dicts for download_features()
    batch_ids     = {row['product_id'] for row in batch}
    batch_features = [f for f in catalogue_raw if f['Id'] in batch_ids]

    if len(batch_features) != len(batch):
        print(f'WARNING: only {len(batch_features)} / {len(batch)} features found in catalogue cache.')

    # download_features(features, path, options), credentials go in the options dict
    download_options = {'credentials': credentials}

    downloaded = 0
    errors     = []

    # Iterate over features and download each one, with error handling
    for feat in batch_features:
        name = feat.get('Name', feat.get('Id', 'unknown'))
        print(f'  [{downloaded+1}/{len(batch_features)}] {name} ...', end=' ', flush=True)
        try:
            list(download_features([feat], SLC_DIR, download_options))
            downloaded += 1
            print('OK')
        except Exception as e:
            errors.append({'name': name, 'error': str(e)})
            print(f'ERROR: {e}')

    print(f'\nDownloaded: {downloaded}  Errors: {len(errors)}')
    if errors:
        print('\nFailed downloads:')
        for e in errors:
            print(f'  {e["name"]}: {e["error"]}')

Starting download of 11 scenes to ..\data\Sentinel_S1_SLC...
This will take several hours. Monitor disk space during download.

  [1/11] S1A_IW_SLC__1SDV_20201010T171554_20201010T171621_034737_040C09_8033.SAFE ... OK
  [2/11] S1A_IW_SLC__1SDV_20201022T171529_20201022T171556_034912_04121B_799E.SAFE ... OK
  [3/11] S1B_IW_SLC__1SDV_20201028T171459_20201028T171526_024016_02DA63_AF86.SAFE ... OK
  [4/11] S1B_IW_SLC__1SDV_20210625T171500_20210625T171527_027516_0348DE_2B18.SAFE ... OK
  [5/11] S1A_IW_SLC__1SDV_20210806T171533_20210806T171600_039112_049D8B_73E3.SAFE ... OK
  [6/11] S1A_IW_SLC__1SDV_20210911T171559_20210911T171626_039637_04AFA1_7800.SAFE ... OK
  [7/11] S1A_IW_SLC__1SDV_20230820T171545_20230820T171611_049962_0602B3_5AC3.SAFE ... OK
  [8/11] S1A_IW_SLC__1SDV_20230901T171610_20230901T171637_050137_0608B3_D759.SAFE ... OK
  [9/11] S1A_IW_SLC__1SDV_20230913T171545_20230913T171612_050312_060EAC_0770.SAFE ... OK
  [10/11] S1A_IW_SLC__1SDV_20230925T171546_20230925T171613_050487_06149

---
## 9. Verify Downloaded Scenes

In [148]:
# Re-check what is on disk after download
on_disk_after = [row['name'] for _, row in all_ids.iterrows() if is_on_disk(row['name'])]
on_disk_set   = set(on_disk_after)

def _pid_on_disk(pid):
    if not isinstance(pid, str) or not pid:
        return False
    return any(r['name'] in on_disk_set for _, r in all_ids.iterrows() if r['product_id'] == pid)

coverage_df['t1_on_disk'] = coverage_df['slc_t1_id'].apply(_pid_on_disk)
coverage_df['t2_on_disk'] = coverage_df['slc_t2_id'].apply(_pid_on_disk)
coverage_df['t3_on_disk'] = coverage_df['slc_t3_id'].apply(_pid_on_disk)
# t4 only required on disk for quadruplet-valid events; others default True (not needed)
coverage_df['t4_on_disk'] = coverage_df.apply(
    lambda r: _pid_on_disk(r['slc_t4_id']) if r.get('valid_after_quad', False) else True,
    axis=1
)
coverage_df['fully_ready'] = (
    coverage_df['full_triplet_ok'] &
    coverage_df['t1_on_disk'] &
    coverage_df['t2_on_disk'] &
    coverage_df['t3_on_disk'] &
    coverage_df['t4_on_disk']
)

n_ready = coverage_df['fully_ready'].sum()
n_total = len(coverage_df)

print(f'Events fully ready for coherence processing: {n_ready} / {n_total}  ({n_ready/n_total*100:.0f}%)')
if n_ready < n_total:
    not_ready = coverage_df[~coverage_df['fully_ready']]
    print(f'\n{len(not_ready)} events not yet ready (download remaining batches):')
    cols = ['event_date_str', 'full_triplet_ok', 'valid_after_quad',
            't1_on_disk', 't2_on_disk', 't3_on_disk', 't4_on_disk']
    print(not_ready[cols].head(20).to_string(index=False))

# Update coverage file with disk status
coverage_df.to_csv(SLC_COVERAGE_PATH, index=False)
print(f'\nUpdated: {SLC_COVERAGE_PATH}')

Events fully ready for coherence processing: 14 / 92  (15%)

78 events not yet ready (download remaining batches):
 event_date_str  full_triplet_ok  valid_after_quad  t1_on_disk  t2_on_disk  t3_on_disk  t4_on_disk
       20190509             True              True       False       False       False       False
       20190516             True              True       False       False       False       False
       20190523             True              True       False       False       False       False
       20190530             True              True       False       False       False       False
       20190531             True              True       False       False       False       False
       20190605             True              True       False       False       False       False
       20190606             True              True       False       False       False       False
       20190616             True              True       False       False       False       

---
## 10. SAFE File Management

Identifies which SAFE files are **needed** (for non-blacklisted quadruplet events), which are **on disk**, and which are **safe to delete**. Also shows which `match_id`s depend on each file so shared scenes are not accidentally removed before all dependent events are processed.

In [151]:
import glob as _glob_safe

# ── Collect all SAFE files currently on disk ──────────────────────────────────
safe_on_disk = {}
for _p in _glob_safe.glob(os.path.join(SLC_DIR, '*.SAFE.zip')):
    canon = os.path.basename(_p).replace('.zip', '')
    safe_on_disk[canon] = _p
for _p in _glob_safe.glob(os.path.join(SLC_DIR, '*.SAFE')):
    if os.path.isdir(_p):
        canon = os.path.basename(_p)
        safe_on_disk[canon] = _p

# ── Check which match_ids are already fully processed ─────────────────────────
def _match_id_done(mid):
    """True if both coh_before and coh_after TIFs exist for this match_id."""
    before = os.path.join(COH_DIR, f'{mid}_coh_before.tif')
    after  = os.path.join(COH_DIR, f'{mid}_coh_after.tif')
    return os.path.exists(before) and os.path.exists(after)

# ── Build needed set from slc_index_df ────────────────────────────────────────
needed_files = {}

def _register(safe_name, mid, needed_dict):
    if not isinstance(safe_name, str) or not safe_name:
        return
    canon = os.path.basename(safe_name)
    if canon.endswith('.zip'):
        canon = canon[:-4]
    if not canon.endswith('.SAFE'):
        canon = canon + '.SAFE'
    needed_dict.setdefault(canon, set()).add(int(mid))

for _, r in slc_index_df.iterrows():
    mid = int(r['match_id'])
    if mid in SKIP_MATCH_IDS or not r.get('full_triplet_ok', False):
        continue
    for col in ['slc_t1_name', 'slc_t2_name', 'slc_t3_name']:
        _register(r.get(col), mid, needed_files)
    if r.get('valid_after_quad', False):
        _register(r.get('slc_t4_name'), mid, needed_files)

# ── Categorise with processing-aware check ────────────────────────────────────
needed_on_disk      = {}   # needed, present, AND has unprocessed match_ids
processed_on_disk   = {}   # needed, present, but ALL match_ids already done → safe to delete
needed_missing      = {}   # needed but absent
disk_not_needed     = {}   # on disk but not in any event's requirements

for canon, mids in needed_files.items():
    unprocessed = {mid for mid in mids if not _match_id_done(mid)}
    if canon in safe_on_disk:
        if unprocessed:
            needed_on_disk[canon] = {'all': mids, 'unprocessed': unprocessed}
        else:
            processed_on_disk[canon] = mids
    else:
        if unprocessed:
            needed_missing[canon] = {'all': mids, 'unprocessed': unprocessed}
        # If all match_ids are done and file isn't on disk, nothing to do

for canon in safe_on_disk:
    if canon not in needed_files:
        disk_not_needed[canon] = safe_on_disk[canon]

# ── Disk space helpers ────────────────────────────────────────────────────────
def _size_gb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e9
    if os.path.isdir(path):
        return sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fs in os.walk(path) for f in fs
        ) / 1e9
    return 8.0

total_disk_gb = sum(_size_gb(p) for p in safe_on_disk.values())
deletable_gb  = (sum(_size_gb(safe_on_disk[c]) for c in processed_on_disk if c in safe_on_disk)
               + sum(_size_gb(p) for p in disk_not_needed.values()))

# ── Print results ─────────────────────────────────────────────────────────────
print('=' * 72)
print(f'KEEP — needed, on disk, unprocessed match_ids remain  ({len(needed_on_disk)} files):')
for canon, info in sorted(needed_on_disk.items()):
    print(f'  {canon}')
    print(f'    ↳ unprocessed: {sorted(info["unprocessed"])}  '
          f'(also used by done: {sorted(info["all"] - info["unprocessed"])})'
          if info['all'] - info['unprocessed'] else
          f'    ↳ needed by: {sorted(info["unprocessed"])}')

print()
print(f'SAFE TO DELETE — on disk, all match_ids fully processed  ({len(processed_on_disk)} files):')
for canon, mids in sorted(processed_on_disk.items()):
    sz = _size_gb(safe_on_disk[canon]) if canon in safe_on_disk else 0
    print(f'  {canon}  ({sz:.2f} GB)')
    print(f'    ↳ was needed by (all done): {sorted(mids)}')

print()
print(f'SAFE TO DELETE — on disk, not needed by any event  ({len(disk_not_needed)} files):')
for canon, path in sorted(disk_not_needed.items()):
    print(f'  {canon}  ({_size_gb(path):.2f} GB)')

print()
print(f'NEED DOWNLOADING — missing, unprocessed match_ids remain  ({len(needed_missing)} files):')
for canon, info in sorted(needed_missing.items()):
    print(f'  {canon}')
    print(f'    ↳ needed by: {sorted(info["unprocessed"])}')

print()
print('=' * 72)
n_deletable = len(processed_on_disk) + len(disk_not_needed)
print('DISK SPACE SUMMARY:')
print(f'  Files on disk:          {len(safe_on_disk):3d}  ({total_disk_gb:.1f} GB)')
print(f'  Must keep:              {len(needed_on_disk):3d}')
print(f'  Can be deleted:         {n_deletable:3d}  ({deletable_gb:.1f} GB)')
print(f'  Still need downloading: {len(needed_missing):3d}  (~{len(needed_missing) * 8:.0f} GB estimated)')
print('=' * 72)

KEEP — needed, on disk, unprocessed match_ids remain  (0 files):

SAFE TO DELETE — on disk, all match_ids fully processed  (15 files):
  S1A_IW_SLC__1SDV_20200928T171529_20200928T171556_034562_0405EA_B945.SAFE  (7.91 GB)
    ↳ was needed by (all done): [62, 63, 64]
  S1A_IW_SLC__1SDV_20210607T171529_20210607T171556_038237_04832F_01A0.SAFE  (8.10 GB)
    ↳ was needed by (all done): [69, 70, 71, 72]
  S1A_IW_SLC__1SDV_20210619T171530_20210619T171557_038412_048863_1462.SAFE  (8.10 GB)
    ↳ was needed by (all done): [71, 72, 73, 74]
  S1A_IW_SLC__1SDV_20210725T171532_20210725T171559_038937_049829_2575.SAFE  (8.10 GB)
    ↳ was needed by (all done): [75]
  S1A_IW_SLC__1SDV_20210806T171533_20210806T171600_039112_049D8B_73E3.SAFE  (8.10 GB)
    ↳ was needed by (all done): [75]
  S1A_IW_SLC__1SDV_20210818T171533_20210818T171600_039287_04A39F_7CF1.SAFE  (8.10 GB)
    ↳ was needed by (all done): [76, 77, 78]
  S1B_IW_SLC__1SDV_20200910T171459_20200910T171526_023316_02C47B_946A.SAFE  (8.23 GB)
 

---
## Summary of the outputs

- `data/slc_scene_index.csv`: Full triplet plan per event (SLC dates, product IDs, baselines)
- `data/slc_event_coverage.csv`: Per-event readiness status (disk check included)
- `data/Sentinel_S1_SLC/*.SAFE`: Downloaded SLC scenes
- `data/slc_catalogue_cache.json`: CDSE metadata cache (avoids repeated API queries)